# 1.4 m/s — 真の文字拡大版（フォント修正版）

前の `LargeText_RenderOnly.ipynb` には，Colabでフォントが見つからない場合に**小さい標準フォントへ自動的に戻る不具合**がありました．このノートブックは，Matplotlibに同梱されたTrueTypeフォントを明示的に指定し，そのサイズを検査します．

- **コードセルは2つだけ**．上から順に実行してください．CPUランタイムで十分です．Isaac Gymの再実行は不要です．
- 出力は新ファイル `P012_1p4mps_straightness_3panel_fontfixed.mp4`（1920×1080，18秒）．古いMP4は上書きしません．
- 描画中の出力に `FONT_OK` と表示されます．表示されない場合は文字サイズの確認ができていません．
- 方策名は46 px，タイトルは40 px，接地許容幅は28 px，XY・Yawは25 pxに固定されます．


In [ ]:
# Google Drive内のNPZデータから、フォントを確実に拡大した動画を再描画します。
from google.colab import drive
from pathlib import Path
import subprocess, sys, py_compile

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
VIDEO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training/contact_tolerance_P0_P1_P2_from6500_20261024_075342/evaluation/high_speed_limit_P012_coarse_20261024/video_1p4_compare')
assert VIDEO_DIR.is_dir(), f'Google Driveフォルダーがありません: {VIDEO_DIR}'
for name in ('P0','P1','P2'):
    assert (VIDEO_DIR / f'g1_pose_{name}_1p4_rep0.npz').exists(), name
    assert (VIDEO_DIR / f'g1_pose_info_{name}.json').exists(), name

# Matplotlibの同梱TTFを使うので、Colabのシステムフォント構成に依存しません。
subprocess.run([sys.executable,'-m','pip','-q','install','Pillow','imageio','imageio-ffmpeg','matplotlib'],check=True)
RENDER_SCRIPT = Path('/content/g1_1p4_fontfixed_renderer.py')
RENDER_SCRIPT.write_text('"""Render a 1920x1080 typography-improved version from saved Isaac Gym NPZs.\nNo physics simulation or network. Original 1.4 m/s MP4 remains untouched.\n"""\nimport sys\nimport json\nimport math\nimport csv\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\nimport imageio.v2 as imageio\n\nOUT = Path(sys.argv[1])\nPREVIEW_ONLY = \'--preview-only\' in sys.argv[2:]\nFPS = 25\nW, H = 1920, 1080\nPW = W // 3\nPOLICIES = (\'P0\', \'P1\', \'P2\')\nCOLORS = {\'P0\': (39, 110, 205), \'P1\': (225, 111, 28), \'P2\': (207, 56, 71)}\nLEFT, RIGHT = (18, 147, 173), (213, 130, 64)\nTEXT = (35, 48, 68)\nMUTED = (73, 89, 109)\n\n# Matplotlib ships reproducible DejaVu TrueType fonts with Colab.\n# Never silently fall back to ImageFont.load_default(): it ignores requested sizes.\nimport matplotlib\nfont_dir = Path(matplotlib.get_data_path()) / \'fonts\' / \'ttf\'\nregular = font_dir / \'DejaVuSans.ttf\'\nbold = font_dir / \'DejaVuSans-Bold.ttf\'\nassert regular.is_file() and bold.is_file(), f\'Large-text TrueType fonts missing: {font_dir}\'\nF_TITLE = ImageFont.truetype(str(bold), 40)\nF_TIME = ImageFont.truetype(str(bold), 29)\nF_POLICY = ImageFont.truetype(str(bold), 46)\nF_TOL = ImageFont.truetype(str(regular), 28)\nF_SECTION = ImageFont.truetype(str(bold), 26)\nF_VALUE = ImageFont.truetype(str(bold), 25)\nF_LEGEND = ImageFont.truetype(str(regular), 24)\nlabel_bbox = F_POLICY.getbbox(\'P0\')\nlabel_height = label_bbox[3] - label_bbox[1]\nassert label_height >= 30, f\'Incorrect font size: P0 text height {label_height}px\'\nprint(\'FONT_OK:\', str(bold), \'P0 label height(px):\', label_height, flush=True)\n\nD = {}\ninfo = {}\nfor p in POLICIES:\n    path = OUT / f\'g1_pose_{p}_1p4_rep0.npz\'\n    meta_path = OUT / f\'g1_pose_info_{p}.json\'\n    assert path.is_file(), f\'保存済み姿勢データが見つかりません: {path}\'\n    assert meta_path.is_file(), f\'メタデータが見つかりません: {meta_path}\'\n    D[p] = np.load(path)\n    info[p] = json.loads(meta_path.read_text(encoding=\'utf-8\'))\n    assert info[p][\'command_mps\'] == 1.4 and info[p][\'rep\'] == 0\n    assert info[p][\'fps\'] == FPS\n    assert len(D[p][\'times\']) >= 440, (p, len(D[p][\'times\']))\n    assert D[p][\'body_xyz\'].shape[0] == len(D[p][\'times\'])\n    assert len(D[p][\'root_xyz\']) == len(D[p][\'times\'])\nassert len({info[p][\'initial_state_sha256\'] for p in POLICIES}) == 1, \'初期条件の不一致\'\nframes = min(len(D[p][\'times\']) for p in POLICIES)\nfor p in POLICIES:\n    assert np.allclose(D[p][\'times\'][:frames], D[\'P0\'][\'times\'][:frames], atol=1e-4), \'時刻の不一致\'\nprint(\'Saved physics samples:\', {p: len(D[p][\'times\']) for p in POLICIES}, \'common frames:\', frames, flush=True)\n\npaths = {p: D[p][\'root_xyz\'][:, :2] - D[p][\'start_root_xy\'][None, :] for p in POLICIES}\nxall = np.concatenate([paths[p][:,0] for p in POLICIES] + [np.array([0.0])])\nyall = np.concatenate([paths[p][:,1] for p in POLICIES] + [np.array([0.0])])\nxmin, xmax = min(-1.0, float(xall.min())-1), max(28.0, float(xall.max())+1)\nymin, ymax = min(-5.0, float(yall.min())-1), max(5.0, float(yall.max())+1)\nscale = min(515.0 / (xmax-xmin), 303.0 / (ymax-ymin))\nxmid, ymid = (xmin+xmax)/2, (ymin+ymax)/2\n\nfor p in POLICIES:\n    names = info[p][\'body_names\']\n    idx = {name: j for j, name in enumerate(names)}\n    chains = []\n    for side, color in [(\'left\', LEFT), (\'right\', RIGHT)]:\n        leg = [n for n in [\'pelvis\', f\'{side}_hip_yaw_link\', f\'{side}_hip_roll_link\',\n                            f\'{side}_hip_pitch_link\', f\'{side}_knee_link\',\n                            f\'{side}_ankle_pitch_link\', f\'{side}_ankle_roll_link\'] if n in idx]\n        if len(leg) > 1:\n            chains.append(([idx[n] for n in leg], color))\n        arm = [n for n in [\'torso_link\', f\'{side}_shoulder_pitch_link\',\n                            f\'{side}_shoulder_roll_link\', f\'{side}_shoulder_yaw_link\',\n                            f\'{side}_elbow_link\', f\'{side}_wrist_roll_link\'] if n in idx]\n        if len(arm) > 1:\n            chains.append(([idx[n] for n in arm], color))\n    torso = [n for n in [\'pelvis\', \'waist_yaw_link\', \'waist_roll_link\', \'torso_link\'] if n in idx]\n    if len(torso) >= 2:\n        chains.append(([idx[n] for n in torso], (70, 82, 101)))\n    info[p][\'chains\'] = chains\n\n\ndef arrow(dr, x, y, angle, color):\n    ux, uy = math.cos(angle), -math.sin(angle)\n    xx, yy = x+22*ux, y+22*uy\n    nx, ny = -uy, ux\n    dr.line([(x,y), (xx,yy)], fill=color, width=6)\n    dr.polygon([(xx+7*ux, yy+7*uy),\n                (xx-9*ux+6*nx, yy-9*uy+6*ny),\n                (xx-9*ux-6*nx, yy-9*uy-6*ny)], fill=color)\n\n\ndef frame_at(i):\n    t = float(D[\'P0\'][\'times\'][i])\n    im = Image.new(\'RGB\', (W,H), (244,247,251))\n    dr = ImageDraw.Draw(im)\n    dr.rectangle((0,0,W,96), fill=(24,36,54))\n    dr.text((28,20), \'UNITREE G1  /  1.4 m/s  /  Isaac Gym\',\n            fill=(255,255,255), font=F_TITLE)\n    clock = f\'t = {t:05.2f} s  /  18.00 s\'\n    box = dr.textbbox((0,0),clock,font=F_TIME)\n    dr.text((W - 34 - (box[2]-box[0]),32),clock,\n            fill=(224,234,246),font=F_TIME)\n\n    for col,p in enumerate(POLICIES):\n        px = col * PW\n        color = COLORS[p]\n        dr.rounded_rectangle((px+10,108,px+PW-10,1070), radius=16,\n                             fill=(255,255,255), outline=(214,225,235), width=2)\n        dr.rounded_rectangle((px+28,128,px+37,210), radius=3,fill=color)\n        dr.text((px+55,123),p,fill=TEXT,font=F_POLICY)\n        tolerance = {\'P0\':\'0 ms\',\'P1\':\'±20 ms\',\'P2\':\'±40 ms\'}[p]\n        dr.text((px+55,181),f\'Contact tolerance: {tolerance}\',fill=MUTED,font=F_TOL)\n        dr.line((px+30,226,px+PW-30,226),fill=(219,227,235),width=2)\n\n        dr.text((px+38,248),\'TOP VIEW  |  World XY trajectory\',\n                fill=TEXT,font=F_SECTION)\n        dr.rounded_rectangle((px+33,290,px+PW-33,623),radius=10,\n                             fill=(247,250,253),outline=(219,227,236),width=2)\n        ox,oy = px+PW/2, 457\n        def tr(x,y):\n            return (int(ox + scale * (float(x)-xmid)),\n                    int(oy - scale * (float(y)-ymid)))\n        for gx in range(math.floor(xmin/5)*5,math.ceil(xmax/5)*5+1,5):\n            dr.line((tr(gx,ymin),tr(gx,ymax)),fill=(226,232,240),width=2)\n        for gy in range(math.floor(ymin/5)*5,math.ceil(ymax/5)*5+1,5):\n            dr.line((tr(xmin,gy),tr(xmax,gy)),fill=(226,232,240),width=2)\n        a,b=tr(xmin,0),tr(xmax,0)\n        for xx in range(a[0],b[0],21):\n            dr.line((xx,a[1],min(xx+12,b[0]),a[1]),fill=(151,166,186),width=3)\n        curve = paths[p][:i+1]\n        if len(curve)>1:\n            dr.line([tr(x,y) for x,y in curve], fill=color, width=6, joint=\'curve\')\n        x,y = curve[-1]\n        cx,cy = tr(x,y)\n        dr.ellipse((cx-9,cy-9,cx+9,cy+9),fill=color,outline=(255,255,255),width=2)\n        yaw = float(D[p][\'yaw_rad\'][i])\n        arrow(dr,cx,cy,yaw,color)\n\n        # Three concise readouts, rather than one tiny dense line.\n        dr.text((px+35,648),f\'X {float(x):+5.1f} m\',fill=TEXT,font=F_VALUE)\n        dr.text((px+235,648),f\'Y {float(y):+5.1f} m\',fill=TEXT,font=F_VALUE)\n        dr.text((px+430,648),f\'Yaw {math.degrees(yaw):+5.1f}°\',fill=TEXT,font=F_VALUE)\n        dr.line((px+30,700,px+PW-30,700),fill=(219,227,235),width=2)\n\n        dr.text((px+38,715),\'SIDE VIEW  |  Robot heading frame\',\n                fill=TEXT,font=F_SECTION)\n        # Leg-color legend is intentionally large enough for the final film.\n        dr.ellipse((px+42,765,px+59,782),fill=LEFT)\n        dr.text((px+68,755),\'Left\',fill=MUTED,font=F_LEGEND)\n        dr.ellipse((px+187,765,px+204,782),fill=RIGHT)\n        dr.text((px+214,755),\'Right\',fill=MUTED,font=F_LEGEND)\n        dr.rounded_rectangle((px+33,798,px+PW-33,1048),radius=10,\n                             fill=(251,253,254),outline=(226,232,239),width=2)\n        ground=1018\n        dr.line((px+48,ground,px+PW-48,ground),fill=(166,179,194),width=3)\n        pos = D[p][\'body_xyz\'][i]\n        root = D[p][\'root_xyz\'][i]\n        cs,ss = math.cos(yaw),math.sin(yaw)\n        def side_projection(pt):\n            dx,dy = float(pt[0]-root[0]),float(pt[1]-root[1])\n            fwd=cs*dx+ss*dy\n            return (int(px+PW/2+190*fwd),int(ground-190*float(pt[2])))\n        for indices,line_color in info[p][\'chains\']:\n            pts=[side_projection(pos[k]) for k in indices]\n            dr.line(pts,fill=line_color,width=10,joint=\'curve\')\n            for qx,qy in pts:\n                dr.ellipse((qx-6,qy-6,qx+6,qy+6),fill=line_color)\n        if not info[p][\'chains\']:\n            for pt in pos:\n                qx,qy=side_projection(pt)\n                dr.ellipse((qx-5,qy-5,qx+5,qy+5),fill=color)\n    return im\n\n\nfilename=\'P012_1p4mps_straightness_3panel_fontfixed.mp4\'\npreview=OUT/\'P012_1p4mps_fontfixed_preview.png\'\nif PREVIEW_ONLY:\n    for name,j in [(\'start\',0),(\'middle\',frames//2),(\'end\',frames-1)]:\n        dest=OUT/f\'preview_large_text_{name}.png\'\n        frame_at(j).save(dest)\n        print(\'PREVIEW_ONLY:\',dest,flush=True)\nelse:\n    output=OUT/filename\n    with imageio.get_writer(str(output),fps=FPS,codec=\'libx264\',quality=7,macro_block_size=2) as writer:\n        for i in range(frames):\n            im=frame_at(i)\n            if i==min(frames-1,frames//2):\n                im.save(preview)\n            writer.append_data(np.asarray(im))\n            if (i+1)%75==0 or i==0:\n                print(\'Rendered\',i+1,\'/\',frames,flush=True)\n    print(\'VIDEO_DONE:\',output,\'MiB:\',round(output.stat().st_size/1024**2,2),flush=True)\n    print(\'PREVIEW:\',preview,flush=True)\n', encoding='utf-8')
py_compile.compile(str(RENDER_SCRIPT),doraise=True)
result=subprocess.run([sys.executable,str(RENDER_SCRIPT),str(VIDEO_DIR)],capture_output=True,text=True)
print(result.stdout[-6000:])
if result.stderr: print(result.stderr[-3000:])
assert result.returncode==0, f'MP4再描画エラー: {result.returncode}'
assert (VIDEO_DIR / 'P012_1p4mps_straightness_3panel_fontfixed.mp4').is_file()
print('文字拡大版MP4保存先:', VIDEO_DIR / 'P012_1p4mps_straightness_3panel_fontfixed.mp4')


In [ ]:
from google.colab import drive
from IPython.display import display, Image, Video
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
VIDEO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training/contact_tolerance_P0_P1_P2_from6500_20261024_075342/evaluation/high_speed_limit_P012_coarse_20261024/video_1p4_compare')
mp4=VIDEO_DIR / 'P012_1p4mps_straightness_3panel_fontfixed.mp4'
preview=VIDEO_DIR / 'P012_1p4mps_fontfixed_preview.png'
assert mp4.is_file(), '先に上の描画セルを実行してください．'
print('文字拡大版:',mp4,'size MB',round(mp4.stat().st_size/1024**2,2))
if preview.is_file(): display(Image(filename=str(preview)))
display(Video(str(mp4),embed=True))
